# 2. Transformer Engine Layers, and FP8

Notebook 01 built a GPT out of `nn.Linear` layers. Notebook 00 showed that FP8 can make
matrix multiplies substantially faster -- but `nn.Linear` cannot run in FP8 at all.

This notebook replaces the four Linear layers in the block with **Transformer Engine**
equivalents, which can, and then turns FP8 on and measures what it costs and what it buys.

## Objectives

- Swap `nn.Linear` for `te.Linear` -- a one-for-one replacement
- Confirm the model is unchanged: same parameters, same initialization, same loss
- Turn on FP8 with `te.autocast` and measure the speedup on a real model
- Find out whether FP8 costs you any quality
- Compare the FP8 scaling recipes your GPU supports
- Prove FP8 GEMMs actually ran, rather than assuming

## Requirements

- Notebooks 00 and 01
- The dataset (`make data` from the repository root)

## Working through this on your own

The setup cell repeats notebook 01's code so this notebook runs standalone. Section 2.6
trains two models and takes about two minutes; everything else is seconds.

## 2.0 Setup

This is notebook 01's code, unchanged. Read it if you skipped notebook 01; otherwise just
run it.

In [ ]:
import contextlib
import logging
import math
import time
import warnings
import statistics
from dataclasses import dataclass

logging.getLogger("torch._library.opaque_object").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
import torch.nn as nn
import torch.nn.functional as F
import transformer_engine.pytorch as te
from transformer_engine.common import recipe as te_recipe

from helpers import summary, get_tokenizer, get_batch, te_support, warmup
# Import classes built in Notebook 01
from helpers import MiniGPT, TorchBlock

enc = get_tokenizer("data")

@dataclass
class Config:
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    seq_len: int = 1024
    vocab_size: int = 50304

cfg = Config()
BATCH = 24
torch.manual_seed(1337)
print(summary())

## 2.1 Why `nn.Linear` cannot do this

FP8 is not a dtype you can simply ask for. Running a matmul in FP8 means:

1. converting the inputs and weights into 8 bits, which requires choosing a **scale factor**
   that maps their range into E4M3's narrow window,
2. running a GEMM that takes those scales as arguments,
3. keeping the scale factors up to date as training shifts the distributions.

`nn.Linear` does none of this, and PyTorch's autocast will not do it for you. **Transformer
Engine** is NVIDIA's library that does: its layers own their scale factors and dispatch to
the FP8 GEMM when you ask.

`te.Linear` is a drop-in replacement -- same constructor, same call:

```python
nn.Linear(768, 2304)      ->      te.Linear(768, 2304)
```

## 2.2 The swap

Here is `TorchBlock` again with every PyTorch module replaced by its TE equivalent. The
structure is identical; only the class names on the right-hand sides change.

One substitution needs a note. `F.scaled_dot_product_attention` becomes
`te.DotProductAttention`, which wants its inputs as `(batch, seq, heads, head_dim)` --
TE calls this `bshd` -- and applies the causal mask itself. So the two `.transpose(1, 2)`
calls disappear.

(Swapping the attention is *not* what makes FP8 possible -- the four Linear layers are. We
change it here so that this block and notebook 03's fused block differ by exactly one
thing: fusion.)

In [ ]:
class TEUnfusedBlock(nn.Module):
    """Every PyTorch module replaced one-for-one by its Transformer Engine equivalent.

    "Unfused" means the LayerNorms are still separate modules -- that is what notebook 03
    changes. The attention is a fused kernel in both versions.
    """

    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.ln_1 = te.LayerNorm(cfg.n_embd)
        self.qkv  = te.Linear(cfg.n_embd, 3 * cfg.n_embd)
        self.attn = te.DotProductAttention(
            cfg.n_head, cfg.n_embd // cfg.n_head, attention_dropout=0.0,
            qkv_format="bshd", attn_mask_type="causal")
        self.proj = te.Linear(cfg.n_embd, cfg.n_embd)
        self.ln_2 = te.LayerNorm(cfg.n_embd)
        self.fc_1 = te.Linear(cfg.n_embd, 4 * cfg.n_embd)
        self.fc_2 = te.Linear(4 * cfg.n_embd, cfg.n_embd)

    def attention(self, qkv):
        B, T, C3 = qkv.shape
        C = C3 // 3
        q, k, v = (z.view(B, T, self.n_head, C // self.n_head) for z in qkv.split(C, dim=2))
        return self.proj(self.attn(q, k, v).view(B, T, C))

    def forward(self, x):                      # identical to TorchBlock.forward
        x = x + self.attention(self.qkv(self.ln_1(x)))
        return x + self.fc_2(F.gelu(self.fc_1(self.ln_2(x)), approximate="tanh"))


print(TEUnfusedBlock(cfg))

## 2.3 Is it still the same model?

Before measuring anything, check that the swap did not change the network. Three things
should match: the parameter count, how many tensors got the residual-scaled
initialization, and the loss of an untrained model.

The middle one is the easiest to get wrong. The scaled initialization is applied by matching
parameter *names*, and a block type that spells them differently would silently keep the
default scale.

In [ ]:
for name, Block in [("TorchBlock", TorchBlock), ("TEUnfusedBlock", TEUnfusedBlock)]:
    torch.manual_seed(1337)
    m = MiniGPT(cfg, Block).cuda()
    n_resid = sum(1 for n, p in m.named_parameters()
                  if p.dim() >= 2 and n.endswith(MiniGPT.RESIDUAL))
    torch.manual_seed(0)
    x, y = get_batch("train", cfg, 4)
    with torch.autocast("cuda", dtype=torch.bfloat16):
        _, loss = m(x, y)
    print(f"{name:<16}{sum(p.numel() for p in m.parameters())/1e6:8.1f}M params   "
          f"residual-scaled {n_resid}/{2*cfg.n_layer}   untrained loss {loss.item():.3f}")
    del m
    torch.cuda.empty_cache()

Same size, same number of residual-scaled tensors, and untrained losses that agree to
a couple of hundredths.

They are not bit-identical, and the reason is worth knowing. The two blocks call **different
attention kernels** -- PyTorch picks FlashAttention, Transformer Engine prefers cuDNN's fused
attention -- and those accumulate their sums in a different order. Floating-point addition is
not associative, so identical inputs give slightly different outputs. The gap is far smaller
than anything we are about to measure.

> Getting this far took one non-obvious fix. `nn.Linear` initializes its bias randomly, while
> `te.Linear` initializes it to zero -- so simply swapping the layers would have changed the
> model as well as the implementation, and the comparison would have been measuring both.
> `init_weights` zeroes every bias explicitly for that reason. When you swap an
> implementation, the thing to check is not just "does it still run" but "is it still the
> same model".

## 2.4 Does Transformer Engine help on its own?

Before turning on FP8, ask a fair question: is TE faster in BF16, where it has no precision
advantage at all?

In [ ]:
def train(model, cfg, steps=40, batch_size=BATCH, lr=6e-4, fp8=False,
          recipe="delayed", log=False, do_warmup=False):
    """The loop from notebook 01, with one addition: an optional FP8 context."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95),
                            weight_decay=0.1, fused=True)
    model.train()
    if do_warmup: warmup()
    t0, tokens, last = None, 0, float("nan")
    tps = []
    for step in range(steps):
        x, y = get_batch("train", cfg, batch_size)
        # Add FP8 context
        with torch.autocast("cuda", dtype=torch.bfloat16), fp8_context(fp8, recipe):
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if step == 0:
            torch.cuda.synchronize(); t0, tokens = time.perf_counter(), 0
        elif step % 10 == 0:
            torch.cuda.synchronize();
            ttime = time.perf_counter()-t0
            tps.append(tokens/ttime)
            t0, tokens = time.perf_counter(), 0
            torch.cuda.synchronize();
        tokens += x.numel()  
        last = loss.item()
    torch.cuda.synchronize()
    time.sleep(3)
    return {"loss": last, "tok_per_s": statistics.median(tps[-4:])}


def fp8_context(fp8, recipe="delayed"):
    """FP8 wraps the FORWARD pass only.

    TE records the scaling factors it needs during forward and reuses them in backward,
    which runs outside this context. Wrapping `loss.backward()` too is a common mistake.
    """
    if not fp8:
        return contextlib.nullcontext()
    if recipe == "delayed":
        return te.autocast(enabled=True, recipe=te_recipe.DelayedScaling())
    return te.autocast(enabled=True, recipe=te_recipe.Float8CurrentScaling())

results = {}
for name, Block in [("TorchBlock", TorchBlock), ("TEUnfusedBlock", TEUnfusedBlock)]:
    torch.manual_seed(1337)
    m = MiniGPT(cfg, Block).cuda()
    results[name] = train(m, cfg)
    print(f"  {name:<16}{results[name]['tok_per_s']:>10,.0f} tok/s   (BF16)")
    del m; torch.cuda.empty_cache()

r = results["TEUnfusedBlock"]["tok_per_s"] / results["TorchBlock"]["tok_per_s"]
print(f"\nTransformer Engine in BF16: {r:.2f}x")

> Sorry if results don't look representative. Even with long warmups, I've observed irregular results due to power and temperature throttling during testing

Roughly a wash, or a small gain. That is the honest answer, and it is worth sitting with: **swapping to Transformer Engine is not itself a speedup.**
It is what makes the next step possible.

## 2.5 Turning on FP8

One context manager:

```python
with te.autocast(enabled=True, recipe=...):
    logits, loss = model(x, y)
```

Inside it, TE modules run their GEMMs in FP8. Everything else -- the embeddings, the loss,
the optimizer, the master copy of the weights -- stays in BF16 or FP32. **"FP8 training"
never means the whole model is 8-bit.**

The `recipe` decides how scale factors are chosen. `DelayedScaling` keeps a rolling history
of recent maxima, so it costs nothing extra to compute but always reacts to slightly stale
information.

In [ ]:
torch.manual_seed(1337)
m = MiniGPT(cfg, TEUnfusedBlock).cuda()
results["TEUnfusedBlock + FP8"] = train(m, cfg, fp8=True)
del m; torch.cuda.empty_cache()

base = results["TorchBlock"]["tok_per_s"]
print(f"{'variant':<18}{'tok/s':>11}{'vs PyTorch':>12}")
print("-" * 41)
for k in ("TorchBlock", "TEUnfusedBlock", "TEUnfusedBlock + FP8"):
    print(f"{k:<20}{results[k]['tok_per_s']:>11,.0f}{results[k]['tok_per_s']/base:>11.2f}x")

Compare that against the prediction you made in notebook 01. The four GEMMs here are
`768 x 2304`, `768 x 768`, `768 x 3072` and `3072 x 768` -- around the crossover in notebook
00's sweep -- so a modest gain is exactly right. The toy model's 1.5x at `H = 8192` was
never going to transfer.

The gap has a second cause worth naming: a real model spends time on things FP8 does not
touch. The embedding lookup, the attention kernels, the loss, the optimizer step and every
LayerNorm run at unchanged speed. **Notebook 03 explores reducing that overhead.**

## 2.6 Does FP8 cost you quality?

Speed is only half the question. FP8 has about 8 representable values between 1 and 2, where
BF16 has 128 -- so does the model learn worse?

Train both from the same seed on the same data and compare.

In [ ]:
# Three runs, not two. The third is a SECOND BF16 run at a different seed -- without it
# there is nothing to compare the FP8 difference against.
quality = {}
for label, seed, fp8 in [("BF16 seed 1337", 1337, False),
                         ("BF16 seed 1338", 1338, False),
                         ("FP8  seed 1337", 1337, True)]:
    torch.manual_seed(seed)
    m = MiniGPT(cfg, TEUnfusedBlock).cuda()
    quality[label] = train(m, cfg, steps=400, fp8=fp8, do_warmup=False)
    print(f"  {label:<16}final loss {quality[label]['loss']:.4f}   "
          f"{quality[label]['tok_per_s']:>9,.0f} tok/s")
    del m; torch.cuda.empty_cache()

effect = quality["FP8  seed 1337"]["loss"] - quality["BF16 seed 1337"]["loss"]
noise  = quality["BF16 seed 1338"]["loss"] - quality["BF16 seed 1337"]["loss"]

print(f"\n  FP8 - BF16   (same seed, precision differs): {effect:+.4f}   <- the effect")
print(f"  BF16 - BF16  (same precision, seed differs): {noise:+.4f}   <- the noise floor")
print(f"\n  |effect| / |noise| = {abs(effect) / abs(noise):.1f}x")

if abs(effect) < 2 * abs(noise):
    print("\n  The effect is NOT distinguishable from noise at this run length.")
    if effect < 0:
        print("  Note the sign: FP8 scored BETTER than BF16 here. Fewer bits did not")
        print("  improve the model -- that is what a result inside the noise looks like,")
        print("  and it is why the sign of a too-small difference tells you nothing.")
else:
    print("\n  The effect is larger than the noise floor, so it is worth taking seriously.")

**This is the single most important habit in the notebook.** A difference between two
runs means nothing until you know how much two runs *of the same thing* differ. So we spend
a third training run establishing that, and compare against it.

At 300 steps, the noise floor is large. The model is nowhere near converged, and it is still in the steep part of its loss curve where small perturbations move the number a lot.
The ratio above should only be treated as a demonstration of the method.
If the effect is not clearly bigger than the noise, you have not measured anything.

The reference sweep for this workshop, trained to convergence at a fixed seed, puts the real figure at about **0.007** validation loss against a same-seed rerun floor of about 0.002 (3x the noise).
While this does mean that FP8 does cost a little quality, the speedup is usually worth it.

## 2.7 Which recipes does your GPU support?

"FP8" is not one thing. Support is patchy and is **not** simply a matter of how new the GPU
is, so ask at runtime rather than assuming.

In [ ]:
for name, (ok, why) in te_support().items():
    print(f"  {name:14s} {'yes' if ok else 'no '}  {why[:64]}")

available = [n for n, (ok, _) in te_support().items()
             if ok and n in ("delayed", "current")]
print(f"\ncomparing the per-tensor recipes available here: {available}\n")

for r in available:
    torch.manual_seed(1337)
    m = MiniGPT(cfg, TEUnfusedBlock).cuda()
    out = train(m, cfg, fp8=True, recipe=r)
    print(f"  {r:<10}{out['tok_per_s']:>10,.0f} tok/s   loss {out['loss']:.3f}")
    del m; torch.cuda.empty_cache()

`delayed` scaling takes its scale from a history of recent maxima -- no extra pass over
the tensor, but always slightly behind. `current` scaling computes the scale from the tensor
being converted right now -- more accurate, but it reads the tensor an extra time.

If your GPU reports `block` or `mxfp8` as unsupported, that is informative rather than
disappointing: those recipes need compute capability 9.0 and 10.0 respectively. **Support is
per-feature, not per-generation** -- an Ada card like the L40S can train in FP8 through
Transformer Engine while lacking block scaling entirely.

## 2.8 Proving FP8 actually ran

Everything above assumed the FP8 context did something. It might not have: TE can fall back
to BF16 silently if a shape is unsupported or a recipe is unavailable, and you would see a
slightly disappointing speedup rather than an error.

The only way to tell if FP8 kernels ran is to profile the GPU and look for two types of kernels:

- **quantize/cast kernels** (`__nv_fp8`, `e4m3`, `cast`) -- real tensors were converted to
  FP8. This is proof.
- **recipe bookkeeping** (`amax`, `scale_update`) -- the scaling machinery ran. This fires
  whenever an FP8 context is open, *even if no GEMM used FP8*, so on its own it proves
  nothing.

We're going to do this with PyTorch Profiler, but you can also use something like [NVIDIA Nsight Systems](https://docs.nvidia.com/nsight-systems/UserGuide/index.html#python-profiling)

In [ ]:
import os

from torch.profiler import ProfilerActivity, profile


@contextlib.contextmanager
def quiet_stderr():
    """The CUDA profiler prints status lines straight to the OS stderr, below Python's
    reach. Redirect the file descriptor itself for the duration."""
    saved, devnull = os.dup(2), os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(saved)
        os.close(devnull)


def kernels_for(model, fp8):
    """The set of CUDA kernels that actually ran during one forward pass."""
    x, y = get_batch("train", cfg, 4)
    for _ in range(3):                                  # warm up
        with torch.autocast("cuda", dtype=torch.bfloat16), fp8_context(fp8):
            model(x, y)
    torch.cuda.synchronize()
    with quiet_stderr(), profile(activities=[ProfilerActivity.CUDA]) as prof:
        with torch.autocast("cuda", dtype=torch.bfloat16), fp8_context(fp8):
            model(x, y)
        torch.cuda.synchronize()
    return {e.key for e in prof.key_averages() if e.self_device_time_total > 0}


torch.manual_seed(1337)
m = MiniGPT(cfg, TEUnfusedBlock).cuda()
bf16_k, fp8_k = kernels_for(m, fp8=False), kernels_for(m, fp8=True)
new = fp8_k - bf16_k

QUANTIZE = ("__nv_fp8", "e4m3", "e5m2", "cast_fp8", "quantize")
BOOKKEEPING = ("amax", "scale_update")
quant = [k for k in new if any(s in k.lower() for s in QUANTIZE)]
book  = [k for k in new if any(s in k.lower() for s in BOOKKEEPING) and k not in quant]

print(f"kernels only in the FP8 run: {len(new)}")
print(f"  quantize/cast (proof)     : {len(quant)}")
print(f"  recipe bookkeeping (weak) : {len(book)}\n")
for k in sorted(quant)[:4]:
    print("  QUANTIZE:", k[:96])
print()
print("VERDICT:", "FP8 GEMMs ran." if quant else
      "NO FP8 quantization found -- this run was not using FP8.")
del m; torch.cuda.empty_cache()

Run this whenever a speedup disappoints you. This is the only way to confirm that FP8 actually ran.

## 2.10 What you have

- `te.Linear` is a drop-in replacement for `nn.Linear` that *can* run in FP8
- Swapping to TE alone is roughly speed-neutral -- it is an enabler, not a win
- FP8 buys a real but modest speedup on a 124M model, as notebook 00 predicted
- It costs a small amount of quality; how small needs a noise floor to judge
- Recipe support is per-feature, not per-generation
- You can verify that FP8 ran instead of hoping with profilers

Notebook 03 goes after the overhead FP8 cannot touch: the LayerNorms, and the kernel
launches between them.

## Exercises

1. **Partial swap.** Make a block where only `fc_1` and `fc_2` are `te.Linear` and the
   attention projections stay `nn.Linear`. How much of the FP8 speedup survives? Which
   layers are carrying it?
2. **Shrink the noise floor.** Re-run 2.6 with `steps=1000`. Does the noise floor fall
   faster than the FP8 effect does? How long a run would you need before a 0.007 difference
   became measurable?
3. **Wrap the wrong thing.** Move `loss.backward()` inside the `fp8_context` block. Does it
   error, slow down, or silently do nothing? Explain why.
4. **Shrink it.** Set `n_embd = 256` (and `n_head = 4`) and re-run 2.5. Does FP8 still win?
   Connect your answer to notebook 00 section 0.4.
5. **Verify a negative.** Run the 2.8 check with `fp8=False` on both sides. Confirm it finds
   no quantize kernels -- a test that never fails is not a test.